# 05 - EDA and Feature Engineering**Objective:** Explore bike sharing data patterns and create new features.**Steps:**1. Statistical summary and class distribution2. Visualizations (distributions, correlations, seasonality)3. Feature scaling4. Feature engineering (interaction features)5. Save engineered data

In [ ]:
import pandas as pdimport numpy as npfrom pathlib import Pathimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)print("Libraries imported successfully")

In [ ]:
# TODO: Load the clean data and inspect its structure# Start by reading data/processed/clean_data.csv into a DataFrame.# Then use .info() to see column dtypes and non-null counts,# .head() to preview a few rows, and .describe() for summary statistics.PROCESSED_DIR = Path("../data/processed")# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")# print(df.info())# print(df.head())# print(df.describe())

### Statistical Summary & DistributionBefore building any models, it is important to understand the range and spread of your data.The target variable — `Rented Bike Count` — is continuous (a regression problem).Check its distribution to see if it is skewed, which might affect model performance.

In [ ]:
# TODO: Examine the target variable distribution# Use .describe() on the "Rented Bike Count" column to see min, max, mean, and quartiles.# Then plot a histogram with sns.histplot() or plt.hist() to visualize the shape.# A right-skewed distribution (long tail of high rental counts) is common for count data.# plt.title("Distribution of Rented Bike Count")# plt.xlabel("Rented Bike Count")# plt.show()

### VisualizationsVisual exploration helps you spot patterns and relationships that summary statistics miss.Focus on:- How each feature is distributed (histograms)- Which features correlate with the target (heatmap)- How demand varies by hour of day (seasonality)

In [ ]:
# TODO: Plot histograms for key numerical features# Pick a handful of features that are likely to influence bike rentals:# Hour, Temperature, Humidity, Wind speed, Solar Radiation. Store them in a list called features_to_plot.# Use df[features].hist() with bins=30 and a large figure size to see all distributions side by side.# features_to_plot = [...]# df[features_to_plot].hist(bins=30, figsize=(15, 10))# plt.tight_layout()# plt.show()

In [ ]:
# TODO: Create a correlation heatmap# Use df.corr() to compute pairwise correlations between numeric columns.# Then visualize with sns.heatmap() — this quickly shows which features# are strongly (positively or negatively) correlated with the target.## TODO: Identify the top features correlated with Rented Bike Count# Sort the correlation values for the target column to see which features# have the strongest relationship with bike demand.# plt.figure(figsize=(12, 10))# sns.heatmap(df.corr(), annot=False, cmap="coolwarm", center=0)# plt.title("Feature Correlation Heatmap")# plt.show()## target_corr = df.corr()["Rented Bike Count"].sort_values(ascending=False)# print("Top correlations with Rented Bike Count:")# print(target_corr)

#### A little primer on groupby - `groupby` is a powerful pandas method that allows you to split your data into groups based on some criteria, apply a function to each group, and then combine the results. For example, to see how bike rentals vary by hour of the day, you can do:```pythonhourly_demand = df.groupby("Hour")["Rented Bike Count"].mean()```- `aggregate` is a method that allows you to apply multiple functions to your grouped data. For example, to get both the mean and standard deviation of bike rentals by hour, you can do:```pythonhourly_stats = df.groupby("Hour")["Rented Bike Count"].aggregate(["mean", "std"])```Aggregate functions can be any function that takes a Series and returns a single value, such as `mean`, `std`, `min`, `max`, etc.Aggregate can be deployed on multiple columns at once, and you can specify different functions for each column if needed.

In [ ]:
# === Executed Example: GroupBy and Aggregate ===# Small inline dataset showing how groupby splits data, applies a function,# and combines results — here grouped by Hour to see demand patterns.import pandas as pddata = pd.DataFrame({    "Hour": [0, 0, 6, 6, 12, 12, 18, 18],    "Rented Bike Count": [50, 70, 200, 180, 500, 450, 300, 350],})hourly_mean = data.groupby("Hour")["Rented Bike Count"].mean()print("Average rentals by hour:\n", hourly_mean)hourly_stats = data.groupby("Hour")["Rented Bike Count"].agg(["mean", "std", "min", "max"])print("\nMultiple statistics:\n", hourly_stats)

In [ ]:
# === Commented Template: GroupBy and Aggregate ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "group_col": [val1, val1, val2, val2],#     "value_col": [10, 20, 30, 40],# })# grouped = data.groupby("group_col")["value_col"].mean()# stats = data.groupby("group_col")["value_col"].agg(["mean", "std", "min", "max"])

### Missing Value Imputation

Our clean data has no missing values, but the corrupted variants from
`data_injection/` do. Common strategies:

- **Drop rows**: `df.dropna()` — fast, loses samples
- **Mean/Median imputation**: `SimpleImputer(strategy='median')` — preserves sample count
- **KNN imputation**: `KNNImputer()` — estimates from neighbors, more robust
- **Forward fill**: `df.ffill()` — for sequential data

In [ ]:
# === Executed Example: Missing Value Imputation ===
# Using SimpleImputer to fill missing values with the mean.

from sklearn.impute import SimpleImputer

data = pd.DataFrame({
    "Temperature": [15.0, np.nan, 25.0, 30.0, np.nan],
    "Humidity": [30.0, 40.0, np.nan, 60.0, 70.0],
})

print("Before imputation:")
print(data)

imputer = SimpleImputer(strategy='mean')
imputed = imputer.fit_transform(data)
imputed_df = pd.DataFrame(imputed, columns=data.columns)
print("\nAfter imputation (mean strategy):")
print(imputed_df)
print(f"\nImputed Temperature: {imputer.statistics_[0]:.3f}")
print(f"Imputed Humidity: {imputer.statistics_[1]:.3f}")

In [ ]:
# TODO: Analyze hourly demand patterns# Group the data by Hour and calculate the average Rented Bike Count for each hour.# This reveals the daily demand cycle — e.g., morning and evening rush hour peaks.## hourly_avg = df.groupby ...## sns.lineplot(x=hourly_avg.index, y=hourly_avg.values)# plt.title("Average Bike Rentals by Hour of Day")# plt.xlabel("Hour")# plt.ylabel("Average Rented Bike Count")# plt.show()

### Feature ScalingMany machine learning algorithms (SVR, linear models, neural networks) are sensitive tothe scale of input features. StandardScaler transforms each feature to have mean 0 andstandard deviation 1, which puts all features on equal footing.Tree-based models (Random Forest, XGBoost) do not require scaling since they split onthresholds independently of feature magnitude.

In [ ]:
# A small Scaling Example:from sklearn.preprocessing import StandardScalerimport pandas as pddf = pd.DataFrame(    {        "Temperature": [15, 20, 25, 30, 35],        "Humidity": [30, 40, 50, 60, 70],        "Rented Bike Count": [100, 200, 300, 400, 500],    })scaler = StandardScaler()scaled_features = scaler.fit_transform(df[["Temperature", "Humidity"]])scaled_df = pd.DataFrame(scaled_features, columns=["Temperature_scaled", "Humidity_scaled"])scaled_df["Rented Bike Count"] = df["Rented Bike Count"]print(scaled_df)

In [ ]:
from sklearn.preprocessing import StandardScaler# TODO: Separate features and target, then scale the features# Define X as all columns except "Rented Bike Count" and y as the target column - use the .drop() method.## Create a StandardScaler, call fit_transform() on X to compute the mean and std# and return the scaled array in one step.## After scaling, verify that each feature has mean ~0 and std ~1.# Use np.allclose() or just print the first few mean/std values.## print(f"Mean after scaling (first 5): {X_scaled.mean(axis=0)[:5]}")# print(f"Std after scaling (first 5): {X_scaled.std(axis=0)[:5]}")# print(f"All means near zero: {np.allclose(X_scaled.mean(axis=0), 0, atol=1e-10)}")

### Feature EngineeringNew features derived from existing columns can capture interactions and non-linear relationships.Good candidates for bike sharing:- **Temperature × Humidity**: A "feels-like" temperature — hot + humid is uncomfortable for biking- **Solar Radiation / Rainfall**: A "pleasant weather" index (rain cancels out the sun)Be careful with division by zero — add a small epsilon or +1 to the denominator.#### NoteIn pandas, you can create interaction features like this:```pythondf["feature1_feature2"] = df["feature1"] * df["feature2"]```

In [ ]:
# === Executed Example: Interaction Features ===# Multiplication and ratio on a small inline dataset.import pandas as pddata = pd.DataFrame({    "Temperature": [15, 20, 25, 30, 35],    "Humidity": [30, 40, 50, 60, 70],    "Solar Radiation": [1.5, 2.0, 2.5, 3.0, 0.5],    "Rainfall": [1.0, 1.0, 0.0, 0.0, 3.0],})data["Temp_Humidity"] = data["Temperature"] * data["Humidity"]print("Temperature * Humidity:\n", data[["Temperature", "Humidity", "Temp_Humidity"]])EPS = 1e-6data["Solar_Rain_Ratio"] = data["Solar Radiation"] / (data["Rainfall"] + EPS)print("\nSolar / Rain ratio:\n", data[["Solar Radiation", "Rainfall", "Solar_Rain_Ratio"]])

In [ ]:
# === Commented Template: Interaction Features ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "feature_a": [val1, val2, val3],#     "feature_b": [val1, val2, val3],# })# data["a_times_b"] = data["feature_a"] * data["feature_b"]# EPS = 1e-6# data["a_over_b"] = data["feature_a"] / (data["feature_b"] + EPS)

In [ ]:
# TODO: Create interaction and ratio features# Multiply Temperature by Humidity to capture the combined discomfort effect.## Create a solar-to-rain ratio. Add 1 to rainfall so the ratio stays finite:## TODO: Verify the new features# Check that they have finite values and reasonable ranges.# Hint: Use .describe() or check for inf values with np.isfinite().

In [ ]:
# TODO: Create lag features for the target variable# Bike sharing demand at hour t is often correlated with demand at t-1# (people follow similar daily routines). A 1-hour lag captures this.# For daily cycles, also try shift(24).## df["Rented_Bike_Count_Lag1"] = df["Rented Bike Count"].shift(1)# df["Rented_Bike_Count_Lag24"] = df["Rented Bike Count"].shift(24)## TODO: Handle NaN values introduced by shifting# The first rows after shift(1) will be NaN.# Use .fillna(0) or .dropna() depending on your strategy.# print(f"NaN after lag: {df['Rented_Bike_Count_Lag1'].isna().sum()}")

In [ ]:
# TODO: Save the engineered data for the next notebook# Include both the original and new features.PROCESSED_DIR = Path("../data/processed")PROCESSED_DIR.mkdir(parents=True, exist_ok=True)# df.to_csv(PROCESSED_DIR / "engineered_data.csv", index=False)# print("Engineered data saved to data/processed/engineered_data.csv")

### Unsupervised Clustering (KMeans)

Clustering groups observations without using the target labels.
We use **KMeans** which partitions data into $k$ clusters by minimizing
within-cluster variance.

**Questions:**
- Do the clusters found by KMeans align with the actual Rented Bike Count classes?
- How many natural groups exist in the data?


In [ ]:
# TODO: Apply KMeans clustering and compare with target labels
# Clustering groups data without using labels.
# Use the elbow method to find optimal k, then compare clusters vs target.

# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import adjusted_rand_score
# from sklearn.decomposition import PCA

# Step 1: Scale features
# X_clust = df.drop("Rented Bike Count", axis=1).select_dtypes(include=[np.number])
# X_clust_scaled = StandardScaler().fit_transform(X_clust)
#
# Step 2: Elbow method for k=2..10
# inertias = []
# for k in range(2, min(11, X_clust_scaled.shape[1] + 1)):
#     kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
#     kmeans.fit(X_clust_scaled)
#     inertias.append(kmeans.inertia_)
#
# Step 3: Plot elbow curve
# plt.plot(range(2, min(11, X_clust_scaled.shape[1] + 1)), inertias, 'bo-')
# plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow Method')
# plt.show()
#
# Step 4: Fit KMeans and compare with target
# df["Cluster"] = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_clust_scaled)
# ari = adjusted_rand_score(df["Rented Bike Count"], df["Cluster"])
# print(f"Adjusted Rand Index: {ari:.4f}")
# print(pd.crosstab(df["Cluster"], df["Rented Bike Count"]))
#
# Step 5: Visualize via PCA
# pca_vis = PCA(n_components=2, random_state=42)
# X_pca_vis = pca_vis.fit_transform(X_clust_scaled)
# plt.scatter(X_pca_vis[:, 0], X_pca_vis[:, 1], c=df['Cluster'], cmap='viridis', edgecolors='k', alpha=0.7)
# plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('KMeans Clusters')
# plt.show()


### Principal Component Analysis (PCA)

PCA finds orthogonal axes (principal components) that capture the maximum variance
in the data. It is useful for:
- **Dimensionality reduction**: compressing many features into fewer components
- **Visualization**: projecting high-dimensional data to 2D or 3D
- **Noise reduction**: discarding low-variance components

PCA is **unsupervised** — it does not use the target labels.


In [ ]:
# TODO: Apply PCA for dimensionality reduction and visualization
# PCA finds the axes of maximum variance in the data.
# It can reduce high-dimensional data to 2D for visualization.

# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler

# X = df.drop("Rented Bike Count", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
#
# Step 1: Fit PCA with min(n_features, 10) components
# n_comps = min(X.shape[1], 10)
# pca = PCA(n_components=n_comps, random_state=42)
# X_pca = pca.fit_transform(X_scaled)
#
# Step 2: Scree plot
# plt.bar(range(1, n_comps + 1), pca.explained_variance_ratio_)
# plt.xlabel('PC'); plt.ylabel('Explained Variance Ratio')
# plt.title('Scree Plot'); plt.show()
#
# Step 3: Cumulative explained variance
# cumulative = np.cumsum(pca.explained_variance_ratio_)
# plt.plot(range(1, n_comps + 1), cumulative, 'bo-')
# plt.axhline(y=0.95, color='r', linestyle='--', label='95%')
# plt.legend(); plt.show()
#
# Step 4: 2D PCA scatter colored by target
# plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["Rented Bike Count"], cmap="coolwarm", edgecolors="k", alpha=0.7)
# plt.xlabel('PC1'); plt.ylabel('PC2')
# plt.title('PCA Projection'); plt.colorbar()
# plt.show()
#
# Step 5: Inspect feature loadings
# loadings = pd.DataFrame(
#     pca.components_.T[:, :3],
#     index=X.columns,
#     columns=['PC1', 'PC2', 'PC3'],
# )
# print(loadings['PC1'].abs().sort_values(ascending=False).head(5))


### Recursive Feature Elimination (RFE)

RFE recursively removes the least important features, building a model at each step.
It ranks features by importance and finds the optimal subset.

**Benefits:**
- Reduces overfitting by removing noisy features
- Improves model interpretability
- Can speed up training and prediction


In [ ]:
# TODO: Apply RFE for feature selection
# RFE recursively removes the least important features.
# RFECV uses cross-validation to find the optimal subset.

# from sklearn.feature_selection import RFE, RFECV
# from sklearn.linear_model import LogisticRegression

# X = df.drop("Rented Bike Count", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
# y = df["Rented Bike Count"]
#
# Step 1: Fit RFE
# estimator = LogisticRegression(max_iter=1000, random_state=42)
# rfe = RFE(estimator=estimator, n_features_to_select=min(10, X_scaled.shape[1]), step=1)
# rfe.fit(X_scaled, y)
#
# Step 2: Display feature rankings
# ranking_df = pd.DataFrame({
#     'Feature': X.columns,
#     'Rank': rfe.ranking_,
#     'Selected': rfe.support_,
# }).sort_values('Rank')
# print(ranking_df)
#
# Step 3: RFECV for automatic feature count
# rfecv = RFECV(estimator=estimator, step=1, cv=5, scoring='accuracy')
# rfecv.fit(X_scaled, y)
# print(f"Optimal features: {rfecv.n_features_}")
#
# Step 4: Plot CV accuracy vs feature count
# plt.plot(range(len(rfecv.cv_results_['mean_test_score'])),
#          rfecv.cv_results_['mean_test_score'], 'bo-')
# plt.axvline(x=rfecv.n_features_, color='r', linestyle='--')
# plt.title('RFE: Optimal Feature Count'); plt.show()


### Exercises1. **Try different scalers**: Replace `StandardScaler` with `MinMaxScaler` or `RobustScaler` and compare how the scaled distributions look.2. **Create a weekend feature**: Use `pd.to_datetime()` on the Date column (if available) to label weekday vs weekend and visualize the difference in demand.3. **Lag features**: Create a 1-hour lag of Rented Bike Count — does demand in the previous hour predict the current hour?4. **Log transform the target**: If the target is right-skewed, try `np.log1p()` and check if the distribution becomes more normal.5. **Pairplot**: Use `sns.pairplot()` on a subset of features to visualize 2D relationships between features and the target.